# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [1]:
# ============================================================
# PASO 1. INSTALAR LIBRERÍAS E INICIAR SESIÓN EN HUGGING FACE
# ============================================================
#
# A diferencia de las clases anteriores, aquí NO entrenamos con Groq.
# Groq se usa principalmente para inferencia.
#
# Para hacer fine-tuning necesitamos cargar el modelo directamente
# en la GPU de Colab mediante Hugging Face.
#
# Librerías:
# - transformers: cargar modelo y tokenizer.
# - peft: aplicar LoRA.
# - accelerate: administrar el uso de GPU/dispositivo.
# - trl: incluye SFTTrainer para entrenamiento supervisado.
#
# Antes de ejecutar:
# 1. En Colab ve a Entorno de ejecución > Cambiar tipo de entorno.
# 2. Selecciona GPU.
# 3. En Secrets crea una variable llamada HF_TOKEN.
# 4. Pega tu token de Hugging Face.
# 5. Activa el acceso del notebook a ese secreto.

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

# Recuperamos el token sin escribirlo directamente en el notebook.
login(token=userdata.get('HF_TOKEN'))

print("Sesión de Hugging Face iniciada correctamente.")

# Verificación opcional:
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 5.7 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.
GPU disponible: True
GPU: Tesla T4


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [2]:
# ============================================================
# PASO 2. CARGAR EL MODELO BASE Y SU TOKENIZER
# ============================================================
#
# El Hands On utiliza TinyLlama porque es un modelo pequeño
# y permite demostrar LoRA en una GPU gratuita de Colab.
#
# tokenizer:
# Convierte texto a tokens y tokens nuevamente a texto.
#
# modelo:
# Es el modelo causal que generará texto y posteriormente
# recibirá los adapters LoRA.
#
# dtype=torch.float16:
# Reduce memoria al usar precisión de 16 bits.
#
# device_map="auto":
# Permite que Transformers coloque automáticamente el modelo
# en el dispositivo disponible, normalmente la GPU.

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# El notebook oficial también menciona como referencia:
# "meta-llama/Llama-3.2-1B-Instruct"
# pero el Hands On trabaja con TinyLlama para la demostración.

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    dtype=torch.float16,
    device_map="auto"
)

print("Modelo base cargado:", modelo_base)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [3]:
# ============================================================
# PASO 3. CREAR UNA LÍNEA BASE ANTES DEL FINE-TUNING
# ============================================================
#
# Antes de entrenar debemos observar cómo responde el modelo base.
# Así podremos comparar después si el comportamiento cambió.
#
# La función generar_respuesta():
# 1. tokeniza el prompt;
# 2. lo mueve al mismo dispositivo del modelo;
# 3. genera nuevos tokens;
# 4. elimina del resultado los tokens que pertenecían al prompt;
# 5. decodifica únicamente la respuesta nueva.

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):

    # Convertir texto a tensores.
    entrada = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(modelo_a_usar.device)

    # Generar la continuación.
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )

    # Separar únicamente los tokens nuevos generados.
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]

    # Convertir los tokens generados nuevamente a texto.
    texto_generado = tokenizer.decode(
        tokens_nuevos,
        skip_special_tokens=True
    )

    return texto_generado.split("\n")[0].strip()


# Este prompt se utilizará antes y después del ajuste.
prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)

print("Respuesta del modelo base:")
print(respuesta_base)

# IMPORTANTE:
# Guarda mentalmente esta respuesta.
# Después del entrenamiento volveremos a utilizar el mismo prompt.


Respuesta del modelo base:
No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [4]:
# ============================================================
# PASO 4. CREAR EL DATASET DE ENTRENAMIENTO
# ============================================================
#
# Para enseñar un comportamiento mediante fine-tuning necesitamos
# ejemplos del patrón que queremos que el modelo aprenda.
#
# Cada ejemplo contiene:
#
# Cliente: pregunta
# Agente: respuesta esperada
#
# En esta demo se usan únicamente 5 ejemplos.
# Es suficiente para mostrar el procedimiento, pero NO representa
# un dataset de producción.

from datasets import Dataset

ejemplos = [
    {
        "texto":
        "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\n"
        "Agente: Sí, puedes solicitar el cambio dentro de la primera hora "
        "escribiendo a soporte@tienda.com."
    },
    {
        "texto":
        "Cliente: ¿Cuánto tarda el reembolso?\n"
        "Agente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."
    },
    {
        "texto":
        "Cliente: ¿Tienen envío el mismo día?\n"
        "Agente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."
    },
    {
        "texto":
        "Cliente: ¿Puedo pagar en el momento de la entrega?\n"
        "Agente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."
    },
    {
        "texto":
        "Cliente: ¿Cómo rastreo mi paquete?\n"
        "Agente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."
    },
]

# Convertimos la lista de diccionarios al formato Dataset
# utilizado por las herramientas de Hugging Face.
dataset = Dataset.from_list(ejemplos)

dataset


Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [5]:
# ============================================================
# PASO 5. CONFIGURAR Y APLICAR LoRA
# ============================================================
#
# LoRA NO entrena todos los parámetros del modelo.
# Agrega pequeñas matrices entrenables a módulos específicos.
#
# En este Hands On:
#
# r=8
#   Es el rango de las matrices LoRA.
#   Un rango mayor ofrece más capacidad de adaptación,
#   pero aumenta los parámetros entrenables.
#
# lora_alpha=16
#   Controla la escala de la actualización introducida por LoRA.
#
# target_modules=["q_proj", "v_proj"]
#   Indica en qué proyecciones de atención colocaremos LoRA.
#
# lora_dropout=0.0
#   En esta demo no aplicamos dropout a los adapters.
#
# task_type="CAUSAL_LM"
#   Indica que estamos trabajando con un modelo causal de lenguaje.

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed

# Fijar semilla ayuda a que la demo sea más reproducible.
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

# Aplicamos LoRA sobre el modelo base.
modelo_lora = get_peft_model(modelo, config_lora)

# Esta línea muestra cuántos parámetros se entrenarán realmente.
modelo_lora.print_trainable_parameters()

# Idea clave para el alumno:
# el modelo completo tiene muchos parámetros,
# pero LoRA permite entrenar solo una pequeña fracción.


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [6]:
# ============================================================
# PASO 6. CONFIGURAR SFTTrainer Y ENTRENAR
# ============================================================
#
# SFT = Supervised Fine-Tuning.
#
# Aquí definimos cómo se realizará el entrenamiento.
#
# output_dir:
#   Carpeta donde se guardarán los resultados.
#
# num_train_epochs=30:
#   El pequeño dataset se recorrerá 30 veces.
#
# per_device_train_batch_size=5:
#   Se usan los cinco ejemplos por lote en esta demo.
#
# learning_rate=2e-4:
#   Tamaño de los pasos con los que se actualizan los adapters.
#
# logging_steps=1:
#   Registramos el loss en cada paso.
#
# dataset_text_field="texto":
#   Indica qué columna contiene los textos del entrenamiento.
#
# max_length=128:
#   Longitud máxima usada en la preparación de las secuencias.

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

# Aquí ocurre el entrenamiento real de los adapters LoRA.
resultado_entrenamiento = trainer.train()

print("Pérdida promedio/final reportada por train():",
      resultado_entrenamiento.training_loss)

# Durante la ejecución observa cómo cambia el valor de loss.
# En términos generales, una reducción indica que el modelo
# se ajusta mejor a los ejemplos usados para entrenar.


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.554', 'grad_norm': '1.737', 'learning_rate': '0.0002', 'entropy': '2.25', 'num_tokens': '229', 'mean_token_accuracy': '0.5', 'epoch': '1'}
{'loss': '2.512', 'grad_norm': '1.753', 'learning_rate': '0.0001933', 'entropy': '2.244', 'num_tokens': '458', 'mean_token_accuracy': '0.4955', 'epoch': '2'}
{'loss': '2.456', 'grad_norm': '1.926', 'learning_rate': '0.0001867', 'entropy': '2.234', 'num_tokens': '687', 'mean_token_accuracy': '0.5089', 'epoch': '3'}
{'loss': '2.387', 'grad_norm': '2.05', 'learning_rate': '0.00018', 'entropy': '2.223', 'num_tokens': '916', 'mean_token_accuracy': '0.5268', 'epoch': '4'}
{'loss': '2.317', 'grad_norm': '1.984', 'learning_rate': '0.0001733', 'entropy': '2.208', 'num_tokens': '1145', 'mean_token_accuracy': '0.5223', 'epoch': '5'}
{'loss': '2.246', 'grad_norm': '2.011', 'learning_rate': '0.0001667', 'entropy': '2.188', 'num_tokens': '1374', 'mean_token_accuracy': '0.5446', 'epoch': '6'}
{'loss': '2.17', 'grad_norm': '2.193', 'learning_rate': '0.0

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.



In [7]:
# ============================================================
# PASO 7. COMPARAR EL LOSS REGISTRADO AL INICIO Y AL FINAL
# ============================================================
#
# El notebook oficial usa el primer loss almacenado en log_history
# como referencia inicial.
#
# PRECISIÓN IMPORTANTE:
# trainer.state.log_history[0]['loss']
# corresponde al primer paso de entrenamiento registrado.
#
# NO es una evaluación del modelo completamente antes de entrenar.
#
# Aun así, dentro de esta demo permite observar la tendencia
# del entrenamiento desde el primer registro hasta el resultado final.

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")

if perdida_inicial != 0:
    reduccion = (1 - perdida_final / perdida_inicial) * 100
    print(f"Reducción aproximada: {reduccion:.0f}%")

# No interpretes esto como:
# "el modelo ya es mejor para cualquier tarea".
#
# Lo que observamos es que se adaptó mejor al patrón de los datos
# utilizados durante esta pequeña demostración.


Pérdida al inicio del entrenamiento: 2.55
Pérdida final del entrenamiento: 1.77
Reducción aproximada: 31%


In [8]:
# ============================================================
# PASO 8. PROBAR EL MODELO AJUSTADO Y USAR UNA REFERENCIA
# ============================================================
#
# Primero generamos nuevamente una respuesta con TinyLlama + LoRA
# usando EL MISMO prompt de la línea base.
#
# Así podemos observar cualitativamente si cambió su comportamiento.
#
# Después el notebook muestra una respuesta de un modelo mayor
# mediante Groq como referencia de calidad.
#
# IMPORTANTE:
# La respuesta de Groq NO forma parte del entrenamiento LoRA.
# Solo funciona como punto de comparación.

!pip install groq -q

from groq import Groq

# Respuesta después del ajuste.
respuesta_ajustada = generar_respuesta(
    modelo_lora,
    prompt_prueba
)

print("Respuesta del modelo ajustado (TinyLlama + LoRA):")
print(respuesta_ajustada)

# Para ejecutar la siguiente sección necesitas tener también
# GROQ_API_KEY guardada en Colab Secrets.
client = Groq(
    api_key=userdata.get('GROQ_API_KEY')
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_prueba + " Respuesta muy breve y corta."
        }
    ]
)

print("\nRespuesta de referencia (Groq / GPT-OSS-20B):")
print(response.choices[0].message.content)

# Reflexión para el alumno:
# - ¿Cambió la respuesta de TinyLlama después de LoRA?
# - ¿Se parece más al patrón enseñado en el dataset?
# - ¿Por qué no esperamos que 5 ejemplos conviertan a TinyLlama
#   en un modelo con la calidad de uno mucho más grande?


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.7 MB/s eta 0:00:00
Respuesta del modelo ajustado (TinyLlama + LoRA):
Sleep, and the client.

Respuesta de referencia (Groq / GPT-OSS-20B):
Sí, pero antes del despacho.


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.

---
# **EJEMPLO DIDÁCTICO DE CHALLENGE: AJUSTE DE TONO CON LoRA**

> **Importante:** esta sección es un ejemplo para explicar cómo abordar el Challenge.  
> No sustituye las instrucciones exactas que aparezcan en la sección oficial del Challenge.

El Challenge solicita demostrar tres competencias:

1. **Fine-tuning con LoRA:** crear una `LoraConfig` y aplicarla al modelo base.
2. **Dataset propio:** construir pares de entrada/respuesta que representen un tono o formato deseado.
3. **Evaluación cuantitativa:** comparar el `loss` antes y después del ajuste.

En este ejemplo enseñaremos un **tono formal y cordial de atención al cliente**.


## **La idea del ejemplo**

Queremos que, ante mensajes breves o informales del cliente, el agente responda con un patrón consistente:

- saludo o reconocimiento cortés;
- lenguaje formal;
- respuesta clara;
- cierre amable.

No buscamos enseñar información nueva del mundo.  
Buscamos modificar de manera consistente **cómo responde** el modelo.

Ese es precisamente el tipo de caso donde el fine-tuning/LoRA tiene sentido dentro de esta clase.


In [9]:
# ============================================================
# CHALLENGE - PASO 1. CARGAR UN MODELO BASE NUEVO
# ============================================================
#
# Para que el ejemplo sea independiente del entrenamiento anterior,
# volvemos a cargar TinyLlama desde cero.
#
# Así evitamos aplicar un segundo LoRA sobre el modelo que ya fue
# entrenado durante el Hands On.

modelo_challenge = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    dtype=torch.float16,
    device_map="auto"
)

print("Modelo base fresco cargado para el ejemplo del Challenge.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base fresco cargado para el ejemplo del Challenge.


In [10]:
# ============================================================
# CHALLENGE - PASO 2. CONSTRUIR UN DATASET PROPIO DE TONO
# ============================================================
#
# Cada ejemplo enseña el mismo patrón:
# Cliente -> respuesta formal y cordial.
#
# Para el Challenge real, el alumno debe crear SUS propios ejemplos
# siguiendo las instrucciones oficiales.
#
# Aquí usamos seis únicamente como demostración.

ejemplos_tono = [
    {
        "texto":
        "Cliente: No me llegó mi pedido.\n"
        "Agente: Lamento la demora. Con gusto puedo orientarle para revisar el estado de su pedido."
    },
    {
        "texto":
        "Cliente: Quiero cancelar.\n"
        "Agente: Con gusto puedo apoyarle con la cancelación. Primero verificaremos el estado actual de su pedido."
    },
    {
        "texto":
        "Cliente: ¿Cuándo llega?\n"
        "Agente: Con gusto. Permítame indicarle cómo consultar la fecha estimada de entrega de su pedido."
    },
    {
        "texto":
        "Cliente: Me cobraron dos veces.\n"
        "Agente: Lamento el inconveniente. Con gusto revisaremos el cargo duplicado para orientarle sobre los siguientes pasos."
    },
    {
        "texto":
        "Cliente: Ya no quiero este producto.\n"
        "Agente: Entiendo. Con gusto puedo explicarle las opciones disponibles para solicitar un cambio o devolución."
    },
    {
        "texto":
        "Cliente: No puedo entrar a mi cuenta.\n"
        "Agente: Lamento la dificultad. Con gusto puedo orientarle para recuperar el acceso a su cuenta."
    },
]

dataset_tono = Dataset.from_list(ejemplos_tono)

print("Número de ejemplos:", len(dataset_tono))
dataset_tono


Número de ejemplos: 6


Dataset({
    features: ['texto'],
    num_rows: 6
})

## **¿Qué estamos enseñando realmente?**

Observa que las respuestas no son idénticas, pero comparten características:

- “Con gusto...”
- “Lamento...”
- “Permítame...”
- tratamiento formal;
- respuesta breve y orientada a ayudar.

La consistencia del dataset es importante: si unos ejemplos son formales, otros agresivos y otros humorísticos, el patrón que intentamos enseñar queda ambiguo.


In [11]:
# ============================================================
# CHALLENGE - PASO 3. MEDIR LOSS ANTES DEL ENTRENAMIENTO
# ============================================================
#
# Para este ejemplo haremos una medición explícita ANTES de LoRA.
#
# Cada texto se usa como secuencia objetivo:
# el modelo intenta predecir el siguiente token y calculamos su loss.
#
# Luego promediamos el loss de los ejemplos.
#
# Esto es más explícito que usar solamente el primer registro
# de trainer.state.log_history.

import numpy as np

def medir_loss_promedio(modelo_eval, textos):
    modelo_eval.eval()
    losses = []

    for texto in textos:
        entrada = tokenizer(
            texto,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(modelo_eval.device)

        # En un causal LM las etiquetas pueden ser los mismos input_ids.
        # El modelo calcula la pérdida de predicción del siguiente token.
        with torch.no_grad():
            salida = modelo_eval(
                **entrada,
                labels=entrada["input_ids"]
            )

        losses.append(salida.loss.item())

    return float(np.mean(losses))


textos_tono = [ejemplo["texto"] for ejemplo in ejemplos_tono]

loss_antes = medir_loss_promedio(
    modelo_challenge,
    textos_tono
)

print(f"Loss promedio ANTES de LoRA: {loss_antes:.4f}")


Loss promedio ANTES de LoRA: 3.0516


In [12]:
# ============================================================
# CHALLENGE - PASO 4. DEFINIR LA CONFIGURACIÓN LoRA
# ============================================================
#
# El Challenge pide definir rango, alpha y módulos objetivo.
#
# Para mantenernos alineados con el Hands On usaremos:
# r = 8
# alpha = 16
# q_proj y v_proj
#
# En el Challenge oficial los valores deben ser los que indiquen
# sus instrucciones o los que el alumno justifique si se le permite elegir.

config_lora_challenge = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_challenge_lora = get_peft_model(
    modelo_challenge,
    config_lora_challenge
)

modelo_challenge_lora.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [13]:
# ============================================================
# CHALLENGE - PASO 5. ENTRENAR LOS ADAPTERS LoRA
# ============================================================
#
# Repetimos el mismo patrón técnico del Hands On:
# dataset + LoRA + SFTTrainer.
#
# Este entrenamiento es deliberadamente pequeño para una demo.

config_challenge = SFTConfig(
    output_dir="/content/resultados_challenge_tono",
    num_train_epochs=20,
    per_device_train_batch_size=3,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer_challenge = SFTTrainer(
    model=modelo_challenge_lora,
    train_dataset=dataset_tono,
    args=config_challenge,
)

resultado_challenge = trainer_challenge.train()

print("Entrenamiento del ejemplo terminado.")


Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

{'loss': '3.154', 'grad_norm': '1.531', 'learning_rate': '0.0002', 'entropy': '2.691', 'num_tokens': '127', 'mean_token_accuracy': '0.4355', 'epoch': '0.5'}
{'loss': '2.928', 'grad_norm': '1.689', 'learning_rate': '0.000195', 'entropy': '2.449', 'num_tokens': '252', 'mean_token_accuracy': '0.418', 'epoch': '1'}
{'loss': '2.908', 'grad_norm': '1.749', 'learning_rate': '0.00019', 'entropy': '2.601', 'num_tokens': '377', 'mean_token_accuracy': '0.4098', 'epoch': '1.5'}
{'loss': '3.003', 'grad_norm': '1.829', 'learning_rate': '0.000185', 'entropy': '2.508', 'num_tokens': '504', 'mean_token_accuracy': '0.4597', 'epoch': '2'}
{'loss': '2.959', 'grad_norm': '1.917', 'learning_rate': '0.00018', 'entropy': '2.762', 'num_tokens': '630', 'mean_token_accuracy': '0.4228', 'epoch': '2.5'}
{'loss': '2.709', 'grad_norm': '2.284', 'learning_rate': '0.000175', 'entropy': '2.297', 'num_tokens': '756', 'mean_token_accuracy': '0.4878', 'epoch': '3'}
{'loss': '2.684', 'grad_norm': '2.225', 'learning_rate': 

In [14]:
# ============================================================
# CHALLENGE - PASO 6. MEDIR LOSS DESPUÉS DEL ENTRENAMIENTO
# ============================================================
#
# Volvemos a evaluar exactamente los mismos textos.
# Lo único que cambió es que ahora el modelo contiene los
# adapters LoRA entrenados.

loss_despues = medir_loss_promedio(
    modelo_challenge_lora,
    textos_tono
)

print(f"Loss promedio ANTES:   {loss_antes:.4f}")
print(f"Loss promedio DESPUÉS: {loss_despues:.4f}")

if loss_antes != 0:
    mejora = (1 - loss_despues / loss_antes) * 100
    print(f"Reducción del loss: {mejora:.2f}%")

# Interpretación:
# Si el loss disminuye, el modelo predice mejor el patrón presente
# en estos datos de entrenamiento.
#
# Esto NO significa que sea mejor para cualquier pregunta ni que
# haya sido validado para producción.


Loss promedio ANTES:   3.0516
Loss promedio DESPUÉS: 1.3348
Reducción del loss: 56.26%


In [15]:
# ============================================================
# CHALLENGE - PASO 7. COMPARAR UNA RESPUESTA ANTES Y DESPUÉS
# ============================================================
#
# La evaluación cuantitativa principal solicitada es el loss,
# pero también es útil observar cualitativamente el comportamiento.
#
# Probaremos una pregunta NO idéntica a las del dataset.

prompt_nuevo = "Cliente: Mi paquete sigue sin aparecer.\nAgente:"

# Para una comparación limpia cargamos nuevamente un modelo base.
modelo_referencia_base = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    dtype=torch.float16,
    device_map="auto"
)

respuesta_antes = generar_respuesta(
    modelo_referencia_base,
    prompt_nuevo
)

respuesta_despues = generar_respuesta(
    modelo_challenge_lora,
    prompt_nuevo
)

print("PROMPT:")
print(prompt_nuevo)

print("\nANTES DEL AJUSTE:")
print(respuesta_antes)

print("\nDESPUÉS DEL AJUSTE LoRA:")
print(respuesta_despues)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PROMPT:
Cliente: Mi paquete sigue sin aparecer.
Agente:

ANTES DEL AJUSTE:
¿Por qué no aparece?

DESPUÉS DEL AJUSTE LoRA:
Sí, estoy tratando de informarle que su pedido ha sido enviado a la oficina.


## **¿Cómo explicarlo al alumnado?**

El flujo del Challenge puede resumirse así:

**Modelo base → medir loss → dataset propio → configurar LoRA → entrenar → medir loss nuevamente → comparar**

Lo importante no es únicamente ejecutar `trainer.train()`.

El alumno debe poder explicar:

- qué comportamiento quiere enseñar;
- por qué sus ejemplos representan ese comportamiento;
- qué significan `r`, `lora_alpha` y `target_modules`;
- por qué LoRA entrena solo una fracción de los parámetros;
- qué significa que el `loss` disminuya;
- y por qué una reducción del loss no demuestra por sí sola calidad general o preparación para producción.


## **Mini ejercicio para explicar en clase**

Después de ejecutar el ejemplo, pide al alumnado modificar el tono.

Por ejemplo:

- extremadamente conciso;
- formal institucional;
- amable y empático;
- respuesta en formato fijo.

Luego deben:

1. Reescribir los ejemplos para que todos sigan ese mismo patrón.
2. Volver a cargar un modelo base fresco.
3. Medir el loss inicial.
4. Aplicar LoRA.
5. Entrenar.
6. Medir el loss final.
7. Probar un prompt nuevo.
8. Explicar qué cambió.

### Preguntas de comprobación

- ¿Por qué no estamos entrenando todos los parámetros del modelo?
- ¿Qué controla `r`?
- ¿Para qué sirve `lora_alpha`?
- ¿Qué significa `target_modules=["q_proj", "v_proj"]`?
- ¿Por qué el dataset debe ser consistente?
- ¿Qué nos dice el loss?
- ¿Qué NO nos dice el loss?
- ¿Cuál es la diferencia entre pedir un tono por prompt y ajustarlo mediante ejemplos?

### Relación con el Challenge oficial

Este ejemplo está construido alrededor de las tres competencias descritas en el Challenge proporcionado: **configuración y aplicación de LoRA, construcción de un dataset propio de entrada/respuesta y evaluación cuantitativa del ajuste comparando el loss antes y después**.

Para la entrega real, el alumno debe seguir las instrucciones exactas de la sección oficial del Challenge y mantener completa la parte del Hands On realizada durante clase.
